# RPS I — Agents as Classical Simulation

## A toy model for Axtell (2000) and Macy & Willer (2002)

This notebook uses **Rock–Paper–Scissors (RPS)** to introduce the simplest use of an agent-based computational model. The rules and theoretical probabilities are already known. We use computation to generate numerical realizations of the game and compare them with the analytical benchmark.

The notebook also makes the actors behind the aggregate results visible.

## Learning objectives

By the end of the notebook, you should be able to:

1. identify agents, states, behavioral rules, interaction rules, and outcomes;
2. explain Axtell's first use of agent computation as classical simulation;
3. distinguish one simulation run from repeated Monte Carlo realizations;
4. connect aggregate results to the actors and interactions that generated them.

## 1. The analytical benchmark

RPS has three possible moves. Each move defeats one alternative and loses to another:

- Rock defeats Scissors.
- Scissors defeats Paper.
- Paper defeats Rock.

If both players choose randomly and independently, then:

$$P(\text{win})=P(\text{loss})=P(\text{tie})=\frac{1}{3}.$$

The model is completely specified and its expected outcome is known. This is the setting for **Axtell's Use I**: computation produces numerical realizations of a model we already understand.

In [ ]:
from random import Random
import pandas as pd
import matplotlib.pyplot as plt

MOVES = ['Rock', 'Paper', 'Scissors']

# Each ordered pair maps to the points received by Player 1 and Player 2.
PAYOFF = {
    ('Rock', 'Paper'): (0, 1),
    ('Paper', 'Rock'): (1, 0),
    ('Rock', 'Scissors'): (1, 0),
    ('Scissors', 'Rock'): (0, 1),
    ('Paper', 'Scissors'): (0, 1),
    ('Scissors', 'Paper'): (1, 0),
    ('Rock', 'Rock'): (0, 0),
    ('Paper', 'Paper'): (0, 0),
    ('Scissors', 'Scissors'): (0, 0)
}

## 2. Constructing the agents

Each agent is represented as a Python dictionary. The dictionary stores the agent's current **state**.

We distinguish the current move from the decision rule used to select it.

In [ ]:
players = [
    {'name': 'John', 'score': 0, 'move': None, 'decision_rule': 'random'},
    {'name': 'Mary', 'score': 0, 'move': None, 'decision_rule': 'random'}
]

players

The RPS model has seven visible components:

1. **Agent states:** name, score, move, and decision rule.
2. **Agent representation:** Python dictionaries.
3. **Behavioral rule:** select randomly from the available moves.
4. **Environment/rules:** the payoff dictionary.
5. **Interaction:** two agents play against each other.
6. **State updating:** scores change after the game.
7. **Social outcome:** individual results are aggregated.

## 3. Behavioral and interaction rules

The behavioral rule tells an agent how to select a move. The interaction rule applies the payoff structure and updates both agents.

In [ ]:
def choose_move(agent, rng):
    """Apply the agent's behavioral rule."""
    if agent['decision_rule'] == 'random':
        return rng.choice(MOVES)
    raise ValueError(f"Unknown decision rule: {agent['decision_rule']}")


def classify_outcome(points):
    """Describe the outcome from Player 1's perspective."""
    if points == (1, 0):
        return 'win'
    if points == (0, 1):
        return 'loss'
    return 'tie'


def play_game(player1, player2, rng):
    """Let two agents choose, interact, and update their states."""
    player1['move'] = choose_move(player1, rng)
    player2['move'] = choose_move(player2, rng)

    points = PAYOFF[(player1['move'], player2['move'])]
    player1['score'] += points[0]
    player2['score'] += points[1]

    return {
        'player1': player1['name'],
        'move1': player1['move'],
        'player2': player2['name'],
        'move2': player2['move'],
        'points1': points[0],
        'points2': points[1],
        'outcome1': classify_outcome(points)
    }

## 4. One realization

A single execution produces one possible history of the model. It does not describe the distribution of possible outcomes.

In [ ]:
rng = Random(123)
one_game = play_game(players[0], players[1], rng)

one_game

In [ ]:
pd.DataFrame(players)

## 5. Repeated realizations: Monte Carlo simulation

We now repeat the same stochastic model many times. The model itself does not change: only the random realization changes.

In [ ]:
def simulate_two_players(n_games=10_000, seed=123):
    rng = Random(seed)
    agents = [
        {'name': 'John', 'score': 0, 'move': None, 'decision_rule': 'random'},
        {'name': 'Mary', 'score': 0, 'move': None, 'decision_rule': 'random'}
    ]

    history = []
    for game in range(n_games):
        event = play_game(agents[0], agents[1], rng)
        event['game'] = game + 1
        history.append(event)

    return agents, pd.DataFrame(history)


agents, game_history = simulate_two_players()
game_history.head()

The rows preserve the interactions that generated the results. We can aggregate those events and compare the simulated proportions with the theoretical value of one third.

In [ ]:
outcome_summary = (
    game_history['outcome1']
    .value_counts(normalize=True)
    .rename('simulated')
    .reindex(['win', 'loss', 'tie'])
    .to_frame()
)
outcome_summary['theoretical'] = 1 / 3
outcome_summary

In [ ]:
ax = outcome_summary.plot.bar(figsize=(8, 4), ylim=(0, 0.5))
ax.set_title('RPS outcomes: simulation and analytical benchmark')
ax.set_xlabel('Outcome for Player 1')
ax.set_ylabel('Proportion')
ax.tick_params(axis='x', rotation=0)
plt.show()

## 6. One run is not enough

A stochastic model should be replicated. Each replication below contains 300 games. We record John's proportion of wins in every replication.

In [ ]:
replications = []

for seed in range(200):
    _, history = simulate_two_players(n_games=300, seed=seed)
    replications.append({
        'seed': seed,
        'win_proportion': (history['outcome1'] == 'win').mean()
    })

replications = pd.DataFrame(replications)
replications.head()

In [ ]:
ax = replications['win_proportion'].plot.hist(
    bins=15, figsize=(8, 4), edgecolor='white'
)
ax.axvline(1 / 3, color='red', linestyle='--', label='Theoretical value = 1/3')
ax.set_title('Monte Carlo replications of the same RPS model')
ax.set_xlabel("Player 1's proportion of wins")
ax.legend()
plt.show()

The variation comes from stochastic realization, not from changes to the rules. As the number of games increases, simulated proportions should concentrate around the analytical expectation.

## 7. From two players to a population

Axtell represents both individual agents and the agent population. The population procedure below initializes agents, randomly pairs them, lets them interact, and computes statistics after every round.

In [ ]:
def initialize_society(n_agents):
    return [
        {
            'name': f'Agent_{i:02d}',
            'score': 0,
            'move': None,
            'decision_rule': 'random'
        }
        for i in range(n_agents)
    ]


def play_round(society, rng):
    # Random activation prevents a fixed ordering from determining partners.
    order = list(range(len(society)))
    rng.shuffle(order)

    events = []
    for position in range(0, len(order) - 1, 2):
        player1 = society[order[position]]
        player2 = society[order[position + 1]]
        events.append(play_game(player1, player2, rng))

    return events

In [ ]:
rng = Random(123)
society = initialize_society(n_agents=30)
population_history = []

for round_number in range(200):
    events = play_round(society, rng)
    for event in events:
        event['round'] = round_number + 1
        population_history.append(event)

population_history = pd.DataFrame(population_history)
social_outcome = pd.DataFrame(society).sort_values('score', ascending=False)

social_outcome.head(10)

In [ ]:
move_frequencies = (
    pd.concat([population_history['move1'], population_history['move2']])
    .value_counts(normalize=True)
    .reindex(MOVES)
    .rename('simulated proportion')
    .to_frame()
)
move_frequencies['theoretical proportion'] = 1 / 3
move_frequencies

## 8. From factors to actors

The final table contains aggregate attributes such as each agent's score. If we examined only that table, we might look for factors associated with success. But the score of an agent was not produced independently: every point depended on another agent's move.

The event history preserves the generative mechanism:

$$\text{actors} + \text{choices} + \text{interactions} \longrightarrow \text{aggregate outcomes}.$$

This is the connection to **Macy and Willer**. Agent-based modeling does not merely calculate aggregate outcomes; it shows how those outcomes are generated through interactions among actors.

In [ ]:
# Micro-level interactions
population_history.head(10)

In [ ]:
# Macro-level outcome
social_outcome.describe(include='all')

## 9. What this notebook has—and has not—shown

This model contains agents and interactions, but it remains simple:

- agents are autonomous but identical;
- their choices are independent;
- they do not remember previous games;
- they do not learn or adapt;
- the interaction structure does not affect their behavior.

Therefore, RPS I is best understood as **classical simulation implemented with agents**. The next notebook will retain an analytical benchmark while introducing conditions whose dynamics are not completely captured by that benchmark.

## Questions for discussion

1. What information is lost when we retain only the final social-outcome table?
2. Why does a single run provide insufficient evidence about a stochastic model?
3. Are the score differences evidence that agents differ in ability? Why or why not?
4. What would need to change before interaction structure could influence the aggregate result?